# Prepare the Data

In [10]:
import pickle
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import FunctionTransformer, StandardScaler

from sklearn import set_config
set_config(transform_output="pandas")

In [11]:
train_set = pd.read_parquet("train_set.parquet")
X = train_set.drop(columns="early_spring")
y = train_set["early_spring"].astype(int)
X.shape, y.shape

((57, 91), (57,))

In [12]:
# Must be defined before loading preprocessor.pkl in another notebook
class EarlySpringFeatures(BaseEstimator, TransformerMixin):
    def __init__(self, add_deltas=True, add_pre_march_trend=True, combine_oni=True, drop_redundant=True, use_year=False,
                 delta_features=("soil_temp_mean", "farm_avg_temp_mean", "om_temp_mean"),
                 redundant_features=("soil_thawed_days_jan", "soil_thawed_days_feb", "farm_freezing_days_feb")):
        self.add_deltas = add_deltas
        self.add_pre_march_trend = add_pre_march_trend
        self.combine_oni = combine_oni
        self.drop_redundant = drop_redundant
        self.use_year = use_year
        self.delta_features = delta_features
        self.redundant_features = redundant_features

    def fit(self, X, y=None):
        self.feature_names_out_ = self._engineer(X).columns.to_numpy()
        return self

    def get_feature_names_out(self, input_features=None):
        return self.feature_names_out_

    def transform(self, X):
        return self._engineer(X)

    def _engineer(self, X):
        data = X.drop(columns="early_spring", errors="ignore")
        data["phil_pred"] = data["phil_pred"].astype(int)

        if self.add_deltas:
            for name in self.delta_features:
                data[f"{name}_delta"] = data[f"{name}_feb"] - data[f"{name}_jan"]
                
        if self.add_pre_march_trend:
            for prefix in ["farm_avg_temp", "soil_mean_soil_temp", "om_temperature_2m_mean"]:
                d1 = f"{prefix}_d_minus_1"
                d7 = f"{prefix}_d_minus_7"
                if d1 in data.columns and d7 in data.columns:
                    data[f"{prefix}_7d_trend"] = data[d1] - data[d7]

        if self.combine_oni:
            data["oni_winter_avg"] = data[["oni_prev_nov", "oni_prev_dec"]].mean(axis=1)
            data = data.drop(columns=["oni_prev_nov", "oni_prev_dec"])

        if self.drop_redundant:
            data = data.drop(columns=list(self.redundant_features), errors="ignore")

        if self.use_year:
            data["year"] = data.index.to_numpy()

        return data

In [13]:
SKEWED = r"(?:precip|snowfall|snowdepth)"
skewed_columns = make_column_selector(pattern=SKEWED)
other_numeric_columns = make_column_selector(pattern=rf"^(?!phil_pred$)(?!.*{SKEWED}).*$")

log_transformer = FunctionTransformer(np.log1p, np.expm1, feature_names_out="one-to-one")

skewed_pipeline = Pipeline(steps=[
    ("impute", SimpleImputer(strategy="median", add_indicator=True)),
    ("log", log_transformer),
    ("scale", StandardScaler()),
])

numeric_pipeline = Pipeline(steps=[
    ("impute", SimpleImputer(strategy="median", add_indicator=True)),
    ("scale", StandardScaler()),
])

preprocessor = Pipeline(steps=[
    ("features", EarlySpringFeatures()),
    ("columns", ColumnTransformer(
        transformers=[
            ("phil", "passthrough", ["phil_pred"]),
            ("skewed", skewed_pipeline, skewed_columns),
            ("numeric", numeric_pipeline, other_numeric_columns),
        ],
        verbose_feature_names_out=False,
    )),
]).set_output(transform="pandas")
preprocessor

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('features', ...), ('columns', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,add_deltas,True
,add_pre_march_trend,True
,combine_oni,True
,drop_redundant,True
,use_year,False
,delta_features,"('soil_temp_mean', ...)"
,redundant_features,"('soil_thawed_days_jan', ...)"


In [14]:
X_prepared = preprocessor.fit_transform(X, y)
X_prepared.head()

,phil_pred,farm_precip_sum_feb,farm_precip_sum_jan,farm_snowfall_sum_feb,farm_snowfall_sum_jan,farm_snowdepth_mean_feb,farm_snowdepth_mean_jan,farm_snowdepth_d_minus_1,farm_snowdepth_d_minus_2,farm_snowdepth_d_minus_3,...,missingindicator_farm_avg_temp_d_minus_2,missingindicator_farm_avg_temp_d_minus_3,missingindicator_farm_avg_temp_d_minus_4,missingindicator_farm_avg_temp_d_minus_5,missingindicator_farm_avg_temp_d_minus_6,missingindicator_farm_avg_temp_d_minus_7,missingindicator_farm_avg_temp_last_7d_mean,missingindicator_farm_avg_temp_mean_delta,missingindicator_farm_avg_temp_7d_trend,missingindicator_oni_winter_avg
year,,,,,,,,,,,,,,,,,,,,,
1963,0,-0.083588,-1.248973,0.947261,-0.071068,0.565202,0.241708,1.916264,1.894821,2.009116,...,-0.235702,-0.190693,-0.235702,-0.235702,-0.133631,-0.133631,-0.133631,-0.133631,-0.133631,-0.274721
1982,0,0.694303,0.978692,-0.357313,0.904019,-0.126850,1.093221,-0.663267,-0.633543,-0.596425,...,-0.235702,-0.190693,-0.235702,-0.235702,-0.133631,-0.133631,-0.133631,-0.133631,-0.133631,-0.274721
1976,0,-0.120727,0.687879,-0.545414,1.320016,-0.030981,0.387007,-0.663267,-0.633543,-0.596425,...,-0.235702,-0.190693,-0.235702,-0.235702,-0.133631,-0.133631,-0.133631,-0.133631,-0.133631,-0.274721
2006,0,-1.175662,0.681660,0.912899,-1.269920,-0.334717,-1.372062,1.916264,-0.633543,-0.596425,...,-0.235702,-0.190693,-0.235702,4.242641,-0.133631,-0.133631,-0.133631,-0.133631,-0.133631,-0.274721
1953,0,-0.531556,0.027989,-2.367642,-0.811097,-1.212596,-0.982753,-0.663267,-0.633543,-0.596425,...,-0.235702,-0.190693,-0.235702,-0.235702,-0.133631,-0.133631,-0.133631,-0.133631,-0.133631,-0.274721


In [ ]:
with open("preprocessor.pkl", "wb") as f:
    pickle.dump(preprocessor, f)